# Tally small-model: data gen -> LoRA fine-tune -> eval
Runs on free Colab T4. Needs Drive folder `Tally/` (chunks.jsonl, test_set.jsonl). Outputs go to `Tally/out/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os,json
D='/content/drive/MyDrive/Tally'; OUT=D+'/out'; os.makedirs(OUT,exist_ok=True)
print(os.listdir(D))

In [ ]:
!pip -q install -U transformers peft trl datasets accelerate bitsandbytes rank_bm25 sentencepiece
import torch; print(torch.cuda.get_device_name(0), torch.__version__)

## Stage A: synthetic Q&A from the docs (teacher = Qwen2.5-3B-Instruct, grounded in each chunk)

In [ ]:
import json,re,random,torch
from transformers import AutoTokenizer, AutoModelForCausalLM
chunks=[json.loads(l) for l in open(D+'/chunks.jsonl')]
test=[json.loads(l) for l in open(D+'/test_set.jsonl')]
TEACH='Qwen/Qwen2.5-3B-Instruct'
if not os.path.exists(OUT+'/synth.jsonl'):
    tok=AutoTokenizer.from_pretrained(TEACH); tok.padding_side='left'
    m=AutoModelForCausalLM.from_pretrained(TEACH,torch_dtype=torch.float16,device_map='cuda')
    P=("You write training data for a TallyPrime (accounting software) assistant. Using ONLY the passage below, write 3 different question-answer pairs "
       "a TallyPrime user might ask. Questions must be self-contained (no 'the passage'). Answers must be 1-4 sentences, concrete, include exact key presses and menu paths when the passage has them. "
       "Output strictly as lines: Q: ... then A: ...\n\nPassage (from page: {t}):\n{x}")
    def run(batch):
        msgs=[tok.apply_chat_template([{"role":"user","content":P.format(t=c['title'],x=c['text'][:1500])}],tokenize=False,add_generation_prompt=True) for c in batch]
        enc=tok(msgs,return_tensors='pt',padding=True).to('cuda')
        with torch.no_grad(): o=m.generate(**enc,max_new_tokens=420,do_sample=True,temperature=0.5,top_p=0.9)
        return tok.batch_decode(o[:,enc.input_ids.shape[1]:],skip_special_tokens=True)
    rows=[]; B=12
    for i in range(0,len(chunks),B):
        b=chunks[i:i+B]
        for c,out in zip(b,run(b)):
            for q,a in re.findall(r'Q:\s*(.+?)\s*\n+A:\s*(.+?)(?=\n+Q:|\Z)',out,flags=re.S):
                rows.append({'q':q.strip(),'a':a.strip(),'src':c['url'],'cid':c['id']})
        if (i//B)%10==0:
            print(i,len(rows),flush=True)
            with open(OUT+'/synth.jsonl','w') as f:
                for r in rows: f.write(json.dumps(r)+'\n')
    with open(OUT+'/synth.jsonl','w') as f:
        for r in rows: f.write(json.dumps(r)+'\n')
    del m; torch.cuda.empty_cache()
rows=[json.loads(l) for l in open(OUT+'/synth.jsonl')]
print(len(rows))

In [ ]:
# filter: grounded (word overlap with its chunk), sane length, no leakage of test questions
cmap={c['id']:c['text'].lower() for c in chunks}
tq=set(re.sub(r'\W+',' ',t['question'].lower()).strip() for t in test)
def toks(s): return set(re.findall(r'[a-z0-9\+]{3,}',s.lower()))
good=[];seen=set()
for r in rows:
    q,a=r['q'],r['a']
    if not(15<=len(q)<=250 and 25<=len(a)<=700): continue
    if re.search(r'passage|the text|the document|above',q+a,re.I): continue
    nq=re.sub(r'\W+',' ',q.lower()).strip()
    if nq in seen or nq in tq: continue
    at=toks(a); ct=toks(cmap[r['cid']])
    if len(at)==0 or len(at&ct)/len(at)<0.7: continue
    seen.add(nq); good.append(r)
print('kept',len(good),'of',len(rows))
random.seed(0); random.shuffle(good)
with open(OUT+'/train_clean.jsonl','w') as f:
    for r in good: f.write(json.dumps(r)+'\n')

## Stage B: baseline + RAG helper

In [ ]:
from rank_bm25 import BM25Okapi
bm=BM25Okapi([re.findall(r'[a-z0-9\+]+',c['text'].lower()) for c in chunks])
def retrieve(q,k=3):
    s=bm.get_scores(re.findall(r'[a-z0-9\+]+',q.lower()))
    idx=sorted(range(len(s)),key=lambda i:-s[i])[:k]
    return "\n---\n".join(chunks[i]['text'][:900] for i in idx)
SYS="You are a TallyPrime expert assistant. Answer briefly and exactly, with key presses and menu paths."
def score(ans,must):
    a=ans.lower().replace(' ','')
    ok=all(any(alt.lower().replace(' ','') in a for alt in grp) for grp in must)
    return 1 if ok else 0
def evaluate(model,tok,use_rag,name):
    tok.padding_side='left'; res=[]
    for i in range(0,len(test),8):
        b=test[i:i+8]; prompts=[]
        for t in b:
            u=t['question'] if not use_rag else "Reference notes:\n"+retrieve(t['question'])+"\n\nQuestion: "+t['question']
            msgs=[{"role":"system","content":SYS},{"role":"user","content":u}]
            try: p=tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True,enable_thinking=False)
            except TypeError: p=tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
            prompts.append(p)
        enc=tok(prompts,return_tensors='pt',padding=True).to('cuda')
        with torch.no_grad(): o=model.generate(**enc,max_new_tokens=200,do_sample=False)
        outs=tok.batch_decode(o[:,enc.input_ids.shape[1]:],skip_special_tokens=True)
        for t,ans in zip(b,outs): res.append({'id':t['id'],'category':t['category'],'q':t['question'],'ans':ans.strip(),'ok':score(ans,t['must'])})
    json.dump(res,open(OUT+f'/eval_{name}.json','w'),indent=1)
    acc=sum(r['ok'] for r in res)/len(res)
    cats={}
    for r in res: cats.setdefault(r['category'],[]).append(r['ok'])
    print(name,'ACC',round(acc,3),{k:round(sum(v)/len(v),2) for k,v in cats.items()})
    return acc
BASE='Qwen/Qwen3-0.6B'
tok=AutoTokenizer.from_pretrained(BASE)
base=AutoModelForCausalLM.from_pretrained(BASE,torch_dtype=torch.float16,device_map='cuda')
evaluate(base,tok,False,'base'); evaluate(base,tok,True,'base_rag')
del base; torch.cuda.empty_cache()

## Stage C: LoRA fine-tune (Qwen3-0.6B)

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
tr=[json.loads(l) for l in open(OUT+'/train_clean.jsonl')]
def fmt(r): return {"messages":[{"role":"system","content":SYS},{"role":"user","content":r['q']},{"role":"assistant","content":r['a']}]}
ds=Dataset.from_list([fmt(r) for r in tr])
m=AutoModelForCausalLM.from_pretrained(BASE,torch_dtype=torch.float32,device_map='cuda')
cfg=SFTConfig(output_dir='/content/ft',num_train_epochs=3,per_device_train_batch_size=8,gradient_accumulation_steps=2,learning_rate=2e-4,lr_scheduler_type='cosine',warmup_ratio=0.05,logging_steps=20,save_strategy='no',fp16=True,max_length=768,report_to=[])
lc=LoraConfig(r=32,lora_alpha=64,lora_dropout=0.05,target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],task_type='CAUSAL_LM')
t=SFTTrainer(model=m,args=cfg,train_dataset=ds,processing_class=tok,peft_config=lc)
t.train()
t.model.save_pretrained(OUT+'/lora_adapter')

In [ ]:
ft=t.model.merge_and_unload().half().eval()
evaluate(ft,tok,False,'ft'); evaluate(ft,tok,True,'ft_rag')
ft.save_pretrained(OUT+'/tally-qwen3-0.6b-merged'); tok.save_pretrained(OUT+'/tally-qwen3-0.6b-merged')
print('saved')